# Digit Doctor -- The Broken CNN

## MNIST Handwritten Digit Classification

This notebook implements a **Convolutional Neural Network (CNN)** to classify handwritten digits (0-9) using the MNIST dataset.

### Pipeline Overview
1. Data Loading
2. Preprocessing
3. Model Architecture (CNN)
4. Compilation and Training
5. Evaluation and Visualization
6. Save Model

---

> **WARNING:** This code was written by a previous developer. The pipeline compiles and trains without any runtime errors, but the model accuracy is suspiciously low. That is basically random guessing for 10 classes.
>
> **Your mission:** Find the bugs, fix the pipeline, and optimize the model to achieve the highest accuracy possible.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.datasets import mnist

# IMPROVEMENT: Consider importing additional layers for regularization and normalization.

print("Libraries imported successfully!")

## Step 2: Load the MNIST Dataset

In [ ]:
# Load MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Image shape: {X_train[0].shape}")
print(f"Pixel value range: {X_train.min()} to {X_train.max()}")

## Step 3: Visualize Some Samples

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.suptitle("Sample MNIST Digits", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Preprocess the Data

Prepare the data for the CNN model.

In [ ]:
# Reshape data to add channel dimension (required for Conv2D)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# BUG 1: Pixel values are not normalized -- neural networks expect small input values.

# The previous developer wrote: "Pixel values are already small enough"
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

print(f"Preprocessed training shape: {X_train.shape}")
print(f"Pixel value range after preprocessing: {X_train.min()} to {X_train.max()}")
# HINT: Does the printed range look correct for a neural network input?

## Step 5: Build the CNN Model

A standard CNN architecture for image classification.

In [ ]:
model = Sequential([

    # BUG 2: Flatten is placed before convolutional layers -- spatial structure is lost.
    Flatten(input_shape=(28, 28, 1)),

    # BUG 3: A Dense layer is used where a Conv2D layer should be.
    Dense(128),

    # BUG 4: No activation function -- the network cannot learn non-linear patterns.
    Dense(64),

    # BUG 5: Output layer has no softmax activation for probability output.
    Dense(10)
])

# IMPROVEMENT: Consider adding Conv2D, MaxPooling2D, Dropout, and BatchNormalization layers.

model.summary()

## Step 6: Compile the Model

In [ ]:
# BUG 6: mean_squared_error is a regression loss, not suitable for classification.

model.compile(
    optimizer='sgd',                   # IMPROVEMENT: A more adaptive optimizer may converge faster.
    loss='mean_squared_error',         # HINT: What loss function is designed for multi-class classification?
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Step 7: Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=3,            # IMPROVEMENT: Consider whether 3 epochs is sufficient for convergence.
    batch_size=256,      # IMPROVEMENT: A smaller batch size may improve training stability.
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete!")

## Step 8: Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"\n{'='*50}")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)")
print(f"{'='*50}")

if test_accuracy >= 0.90:
    print("\nExcellent! Accuracy >= 90% -- Strong Model!")
elif test_accuracy >= 0.85:
    print("\nGood! Accuracy >= 85% -- Solid Model!")
elif test_accuracy >= 0.75:
    print("\nDecent! Accuracy >= 75% -- Pipeline is working!")
else:
    print("\nModel is still broken -- accuracy is too low. Keep debugging!")

## Step 9: Visualize Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train Loss', marker='o')
ax2.plot(history.history['val_loss'], label='Val Loss', marker='s')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Sample Predictions

In [ ]:
predictions = model.predict(X_test[:10], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if predicted_labels[i] == y_test[i] else 'red'
    ax.set_title(f"Pred: {predicted_labels[i]} | True: {y_test[i]}", color=color, fontsize=11)
    ax.axis('off')

plt.suptitle("Model Predictions (Green = Correct, Red = Wrong)", fontsize=14)
plt.tight_layout()
plt.show()

correct = sum(predicted_labels == y_test[:10])
print(f"\nCorrect predictions: {correct}/10")

## Step 11: Save the Model

**IMPORTANT:** Save your trained model as `model.h5`. This is required for automated evaluation.

Do **NOT** change the filename.

In [ ]:
# ============================================================
#  DO NOT CHANGE THE LINE BELOW -- the filename must be model.h5
# ============================================================
model.save('model.h5')
print("Model saved as 'model.h5'")
print("\nCommit and create a Pull Request to get your score!")